# LiCSBAS Jakarta ROI factor-5 test — gated multi-prime loop handling

**Frame:** `098A_09673_121312` (ascending)  
**Period:** 2020-01-01 to 2023-12-31  
**Pinned COMET-LiCSBAS commit:** `a145e6c4217b29eb7348ea95630454a099809098`  
**Verified against the post-migration downloader:** 28 July 2026

This self-contained version 8 notebook makes one controlled change after the full-frame factor-2 and factor-5 multi-prime runs failed loop closure: it crops the verified GEOC stack to the fixed Jakarta ROI before multilooking. It retains LiCSBAS's recommended `--multi_prime` option, explicitly keeps the standard 1.5-radian loop threshold, archives step-12 diagnostics automatically, and applies a fail-loud largest-connected-component network gate before inversion.

**Controlled test:** the fixed ROI is `[106.45, -6.54, 107.22, -6.01]` (west, south, east, north). This version independently windows each source GeoTIFF onto one shared pixel-aligned ROI grid, validates all 609 UNW/CC pairs, and only then runs `NLOOK = 5` at approximately 553 m. Ordinary per-raster georeferencing metadata is normalized only when the ROI-edge mismatch is at most 0.01 source pixel. One separately gated exception reproduces LiCSAR's 18 September 2023 nearest-grid fix for exactly 221 historical UNWs whose pixel centres span the reference outer bounds; the notebook requires the repaired ROI phase arrays to remain bit-for-bit identical and records every repair in provenance. Any other geometry difference fails loudly. Outputs use `_roi_ml5_mp`; they cannot overwrite full-frame diagnostics or the factor-10 baseline. The standard 1.5-radian loop threshold is not relaxed.

Important operating rules:

1. Colab preserves output text from dead sessions. A cell that appears finished but streams no new output did not rerun.
2. Keep the tab active during the download.
3. Use **Runtime → Run all**. The notebook restores the verified 609-pair Drive archive plus verified ROI and factor-5 checkpoints automatically; otherwise it downloads, validates, crops, and checkpoints the required data.
4. Never combine a velocity file from one inversion with `vstd` from another.
5. Every external command below checks its return code and required outputs before reporting success.
6. Processing stops before step 9 unless every interferogram folder contains both unwrapped phase (`UNW`) and coherence (`CC`).
7. A CPU/standard Colab runtime is sufficient; this workflow does not use a GPU.
8. No between-cell repair, diagnostic-packaging, or renaming code is required: resumption, exact-date labels, `roi_ml5_mp` tags, ROI provenance, resolution diagnostics, network gates, checksums, QC summaries, and archives are built in.
9. Image retention and temporal span are evaluated only on the largest connected component; disconnected dates cannot make the gate pass.
10. The full-frame factor-2 and factor-5 outputs remain preserved as failed diagnostics and are never overwritten.


In [ ]:
# 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2 — Install the pinned COMET fork and dependencies
import os, sys, subprocess, pathlib

LICSBAS_REPO = pathlib.Path('/content/LiCSBAS')
LICSBAS_COMMIT = 'a145e6c4217b29eb7348ea95630454a099809098'
if not LICSBAS_REPO.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/comet-licsar/LiCSBAS.git', str(LICSBAS_REPO)],
        check=True,
    )
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'checkout', '--detach', LICSBAS_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(LICSBAS_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == LICSBAS_COMMIT, (actual_commit, LICSBAS_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'h5py', 'statsmodels', 'astropy', 'cmcrameri', 'bottleneck', 'rasterio'
], check=True)
requirements = LICSBAS_REPO / 'requirements.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('LiCSBAS commit:', actual_commit)


In [ ]:
# 3 — Configuration and command runner
import json, hashlib, datetime, glob, re, math, shutil

FRAME = '098A_09673_121312'
START = '20200101'
END = '20231231'
TEST_START = '20221205'
TEST_END = '20230122'
NLOOK = 5
ROI_BOUNDS = (106.45, -6.54, 107.22, -6.01)  # west, south, east, north
ROI_CROP_SCHEMA = 4
MAX_GRID_NORMALIZATION_PIXELS = 0.01
LEGACY_GRID_ENDPOINT_TOLERANCE_PIXELS = 1e-6
EXPECTED_LEGACY_UNW_REPAIRS = 221
LICSAR_GRID_REPAIR_COMMIT = 'e1cb8724643a664a728d69d43904d70221cca7ba'
LICSAR_GRID_REPAIR_URL = f'https://github.com/comet-licsar/licsar_proc/commit/{LICSAR_GRID_REPAIR_COMMIT}'
GEOC_ROI_NAME = 'GEOC_JakartaROI'
GEOCML_NAME = f'GEOCml{NLOOK}_JakartaROI'
TS_NAME = f'TS_{GEOCML_NAME}_mp'
RUN_TAG = f'roi_ml{NLOOK}_mp'
DATE_TAG = f'{START}_{END}'
PERIOD_LABEL = f'{START[:4]}-{END[:4]}'
RUN_ROLE = 'fixed_jakarta_roi_resolution_test_multi_prime'
LOOP_MODE = 'multi_prime'
LOOP_THRESHOLD_RAD = 1.5
MAX_LOOP_BAD_FRACTION = 0.25
MIN_RETAINED_IMAGE_FRACTION = 0.75
MIN_TEMPORAL_SPAN_FRACTION = 0.90
EXPECTED_COMPLETE_PAIRS = 609
MIN_COMPLETE_PAIRS = 20
RESUME_FROM_VERIFIED_ARCHIVE = True
WORK = pathlib.Path('/content/insar')
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/InSAR_Risk_Zoning_Project')
ARCHIVE = DRIVE_OUT / 'LiCSBAS_archive_2020_2023'
BIN = LICSBAS_REPO / 'bin'
LIB = LICSBAS_REPO / 'LiCSBAS_lib'
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE.mkdir(parents=True, exist_ok=True)
GEOC_ARCHIVE = ARCHIVE / 'GEOC_20200101_20231231_verified_v4.tar.gz'
GEOC_ARCHIVE_SHA = ARCHIVE / 'GEOC_20200101_20231231_verified_v4.sha256'
GEOC_INVENTORY = ARCHIVE / 'GEOC_20200101_20231231_verified_v4_inventory.json'
GEOC_ROI_ARCHIVE = ARCHIVE / f'{GEOC_ROI_NAME}_{DATE_TAG}_checkpoint.tar.gz'
GEOC_ROI_ARCHIVE_SHA = ARCHIVE / f'{GEOC_ROI_NAME}_{DATE_TAG}_checkpoint.sha256'
GEOC_ROI_INVENTORY = ARCHIVE / f'{GEOC_ROI_NAME}_{DATE_TAG}_inventory.json'
GEOC_ROI_PREFLIGHT = ARCHIVE / f'{GEOC_ROI_NAME}_{DATE_TAG}_grid_preflight_schema{ROI_CROP_SCHEMA}.json'
GEOCML_ARCHIVE = ARCHIVE / f'{GEOCML_NAME}_{DATE_TAG}_checkpoint.tar.gz'
GEOCML_ARCHIVE_SHA = ARCHIVE / f'{GEOCML_NAME}_{DATE_TAG}_checkpoint.sha256'
GEOCML_INVENTORY = ARCHIVE / f'{GEOCML_NAME}_{DATE_TAG}_inventory.json'
QC_SUMMARY = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_qc_summary.json'

run_env = os.environ.copy()
run_env['PYTHONPATH'] = str(LIB)

def resolve_script(role, preferred, fragments):
    exact = BIN / preferred
    if exact.exists():
        return exact
    candidates = sorted(
        p for p in BIN.glob('LiCSBAS*.py')
        if all(fragment.lower() in p.name.lower() for fragment in fragments)
    )
    if len(candidates) != 1:
        raise RuntimeError(f'Cannot resolve {role}: {candidates}')
    return candidates[0]

SCRIPTS = {
    'download': resolve_script('download', 'LiCSBAS01_get_geotiff.py', ('01', 'get_geo')),
    'multilook': resolve_script('multilook', 'LiCSBAS02_ml_prep.py', ('02', 'ml')),
    'check': resolve_script('check', 'LiCSBAS11_check_unw.py', ('11', 'check_unw')),
    'loop': resolve_script('loop', 'LiCSBAS12_loop_closure.py', ('12', 'loop')),
    'invert': resolve_script('invert', 'LiCSBAS13_sb_inv.py', ('13', 'sb_inv')),
    'vstd': resolve_script('vstd', 'LiCSBAS14_vel_std.py', ('14', 'vel_std')),
    'geotiff': resolve_script('GeoTIFF export', 'LiCSBAS_flt2geotiff.py', ('flt2geotiff',)),
}

def run_script(role, args, required=()):
    cmd = [sys.executable, str(SCRIPTS[role]), *map(str, args)]
    print('RUN:', ' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=WORK, env=run_env)
    if completed.returncode != 0:
        raise RuntimeError(f'{role} failed with return code {completed.returncode}')
    missing = [str(p) for p in required if not pathlib.Path(p).exists()]
    if missing:
        raise RuntimeError(f'{role} completed but required output is missing: {missing}')
    return completed

PAIR_RE = re.compile(r'^\d{8}_\d{8}$')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def _has_product(folder, kind):
    patterns = (f'*.geo.{kind}.tif', f'*.{kind}')
    return any(
        path.is_file() and path.stat().st_size > 0
        for pattern in patterns for path in folder.glob(pattern)
    )

def inventory_pairs(root):
    root = pathlib.Path(root)
    pair_dirs = sorted(
        path for path in root.iterdir()
        if path.is_dir() and PAIR_RE.fullmatch(path.name)
    ) if root.exists() else []
    missing_unw = [path.name for path in pair_dirs if not _has_product(path, 'unw')]
    missing_cc = [path.name for path in pair_dirs if not _has_product(path, 'cc')]
    incomplete = sorted(set(missing_unw) | set(missing_cc))
    return {
        'root': str(root),
        'pair_folders': len(pair_dirs),
        'complete_pairs': len(pair_dirs) - len(incomplete),
        'missing_unw': missing_unw,
        'missing_cc': missing_cc,
        'incomplete_pairs': incomplete,
    }

def require_complete_pairs(root, label, min_pairs=1):
    inventory = inventory_pairs(root)
    print(label, json.dumps(inventory, indent=2))
    if inventory['pair_folders'] < min_pairs:
        raise RuntimeError(
            f'{label} has only {inventory["pair_folders"]} pair folders; '
            f'at least {min_pairs} are required.'
        )
    if inventory['incomplete_pairs']:
        raise RuntimeError(
            f'{label} contains {len(inventory["incomplete_pairs"])} incomplete pairs. '
            'Do not continue to LiCSBAS step 11.'
        )
    return inventory

print(json.dumps({k: str(v) for k, v in SCRIPTS.items()}, indent=2))


In [ ]:
# 4 — Validate every resolved script and inspect step-02 flags
for role, script in SCRIPTS.items():
    help_run = subprocess.run(
        [sys.executable, str(script), '-h'], cwd=WORK, env=run_env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if help_run.returncode not in (0, 1):
        raise RuntimeError(f'{role} help check failed ({help_run.returncode})')
    print(f'--- {role}: {script.name} ---')
    print('\n'.join(help_run.stdout.splitlines()[:12]))

step02_help = subprocess.run(
    [sys.executable, str(SCRIPTS['multilook']), '-h'], cwd=WORK, env=run_env,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
).stdout
for required_flag in ('-i', '-o', '-n'):
    if required_flag not in step02_help:
        raise RuntimeError(f'Step 02 does not advertise required flag {required_flag}')
print('All command/help checks passed.')


## Automatic recovery after a lost Colab session

When a checksum-verified archive from this notebook exists in Drive, the next cell restores it automatically. Otherwise it starts a fresh download. The older incomplete 5.96 GB archive is deliberately ignored.


In [ ]:
# 5 — Restore only a checksum-verified archive produced by this notebook
RESTORED_GEOC = False
if RESUME_FROM_VERIFIED_ARCHIVE and GEOC_ARCHIVE.exists() and GEOC_ARCHIVE_SHA.exists():
    expected_sha = GEOC_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
    actual_sha = sha256_file(GEOC_ARCHIVE)
    if actual_sha != expected_sha:
        raise RuntimeError('The Drive GEOC archive checksum does not match; refusing to restore it.')
    if not (WORK / 'GEOC').exists():
        subprocess.run(['tar', '-xzf', str(GEOC_ARCHIVE), '-C', str(WORK)], check=True)
    restored_inventory = require_complete_pairs(WORK / 'GEOC', 'Restored GEOC', MIN_COMPLETE_PAIRS)
    if restored_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
        raise RuntimeError(f'Restored GEOC has {restored_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')
    RESTORED_GEOC = True
    print('Restored verified GEOC archive:', GEOC_ARCHIVE)
else:
    print('Verified 6.40 GB archive not found. A fresh migration-safe download will run.')
print('The older incomplete 5.96 GB archive is intentionally ignored.')


In [ ]:
# 6 — Test the migrated service, then download and validate the full GEOC stack
if not RESTORED_GEOC:
    existing_inventory = inventory_pairs(WORK / 'GEOC')
    if existing_inventory['pair_folders']:
        print(
            f'Found {existing_inventory["pair_folders"]} partial/existing pair folders. '
            'Skipping the clean preflight gate and resuming the full download.'
        )
    else:
        print(f'Preflight download: {TEST_START} to {TEST_END}')
        run_script(
            'download', ['-f', FRAME, '-s', TEST_START, '-e', TEST_END, '--n_para', '4'],
            required=[WORK / 'GEOC'],
        )
        require_complete_pairs(WORK / 'GEOC', 'Preflight GEOC', min_pairs=1)
    print(f'Full download/resume: {START} to {END}')
    run_script(
        'download', ['-f', FRAME, '-s', START, '-e', END, '--n_para', '4'],
        required=[WORK / 'GEOC'],
    )

geoc_inventory = require_complete_pairs(WORK / 'GEOC', 'Full GEOC', MIN_COMPLETE_PAIRS)
if geoc_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Full GEOC has {geoc_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')
geoc_inventory.update({
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME,
    'start': START,
    'end': END,
    'licsbas_commit': actual_commit,
})
GEOC_INVENTORY.write_text(json.dumps(geoc_inventory, indent=2), encoding='utf-8')
print('GEOC inventory saved:', GEOC_INVENTORY)


In [ ]:
# 7 — Back up the validated GEOC stack and verify its contents and checksum
if not RESTORED_GEOC:
    partial_archive = GEOC_ARCHIVE.with_suffix(GEOC_ARCHIVE.suffix + '.partial')
    backup = subprocess.run(
        ['tar', '-czf', str(partial_archive), '-C', str(WORK), 'GEOC']
    )
    if backup.returncode != 0 or not partial_archive.exists() or partial_archive.stat().st_size == 0:
        raise RuntimeError('GEOC backup failed; do not continue')
    os.replace(partial_archive, GEOC_ARCHIVE)
listing = subprocess.run(
    ['tar', '-tzf', str(GEOC_ARCHIVE)], check=True,
    text=True, stdout=subprocess.PIPE,
).stdout.splitlines()
archive_unw = {pathlib.PurePosixPath(item).parts[1] for item in listing if item.endswith('.geo.unw.tif') and len(pathlib.PurePosixPath(item).parts) >= 3}
archive_cc = {pathlib.PurePosixPath(item).parts[1] for item in listing if item.endswith('.geo.cc.tif') and len(pathlib.PurePosixPath(item).parts) >= 3}
if archive_unw != archive_cc or len(archive_unw) != geoc_inventory['complete_pairs']:
    raise RuntimeError(
        f'Archive pair mismatch: UNW={len(archive_unw)}, CC={len(archive_cc)}, '
        f'expected={geoc_inventory["complete_pairs"]}'
    )
archive_sha = sha256_file(GEOC_ARCHIVE)
GEOC_ARCHIVE_SHA.write_text(f'{archive_sha}  {GEOC_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED backup: {GEOC_ARCHIVE} ({GEOC_ARCHIVE.stat().st_size / 1e9:.2f} GB, {len(archive_unw):,} complete pairs)')
print('SHA-256:', archive_sha)


In [ ]:
# 8 — Crop the verified GEOC stack to Jakarta, then create the factor-5 multilook checkpoint
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window, bounds as window_bounds, from_bounds as window_from_bounds

geoc_root = WORK / 'GEOC'
geoc_roi_root = WORK / GEOC_ROI_NAME
source_unw = next(iter(sorted(geoc_root.glob('*/*.geo.unw.tif'))), None)
if source_unw is None:
    raise FileNotFoundError('The verified GEOC stack contains no source UNW GeoTIFF.')

with rasterio.open(source_unw) as ref:
    if ref.crs is None or ref.crs.to_epsg() != 4326:
        raise RuntimeError(f'Expected EPSG:4326 GEOC input, found {ref.crs}.')
    west, south, east, north = ROI_BOUNDS
    if not (ref.bounds.left <= west < east <= ref.bounds.right and ref.bounds.bottom <= south < north <= ref.bounds.top):
        raise RuntimeError(f'Fixed Jakarta ROI {ROI_BOUNDS} is not fully inside source bounds {tuple(ref.bounds)}.')
    floating_window = window_from_bounds(west, south, east, north, transform=ref.transform)
    col0 = max(0, math.floor(floating_window.col_off + 1e-9))
    row0 = max(0, math.floor(floating_window.row_off + 1e-9))
    col1 = min(ref.width, math.ceil(floating_window.col_off + floating_window.width - 1e-9))
    row1 = min(ref.height, math.ceil(floating_window.row_off + floating_window.height - 1e-9))
    crop_window = Window(col0, row0, col1 - col0, row1 - row0)
    if crop_window.width <= 0 or crop_window.height <= 0:
        raise RuntimeError(f'Fixed Jakarta ROI produced an empty crop window: {crop_window}.')
    reference_grid = (ref.width, ref.height, ref.crs, ref.transform)
    source_bounds = list(ref.bounds)
    source_shape = [ref.height, ref.width]
    source_res_deg = [abs(ref.res[0]), abs(ref.res[1])]
    roi_transform = ref.window_transform(crop_window)
    effective_roi_bounds = list(window_bounds(crop_window, ref.transform))
    centre_lat = (south + north) / 2
    source_x_m = source_res_deg[0] * 111320.0 * math.cos(math.radians(centre_lat))
    source_y_m = source_res_deg[1] * 111132.0

target_width = int(crop_window.width)
target_height = int(crop_window.height)
target_left, target_bottom, target_right, target_top = effective_roi_bounds

def source_window_for_target(src, source, enforce_limit=True):
    if src.crs != reference_grid[2]:
        raise RuntimeError(f'GEOC raster CRS differs from the fixed ROI target: {source} ({src.crs}).')
    if not (src.transform.a > 0 and src.transform.e < 0):
        raise RuntimeError(f'GEOC raster axis direction is incompatible with the fixed ROI target: {source}; transform={list(src.transform)}.')
    floating_col_off, floating_row_off = (~src.transform) * (target_left, target_top)
    rounded_col_off = round(floating_col_off)
    rounded_row_off = round(floating_row_off)
    window_rounding_errors = [
        abs(floating_col_off - rounded_col_off),
        abs(floating_row_off - rounded_row_off),
    ]
    source_window = Window(rounded_col_off, rounded_row_off, target_width, target_height)
    if not (
        source_window.col_off >= 0 and source_window.row_off >= 0
        and source_window.col_off + source_window.width <= src.width
        and source_window.row_off + source_window.height <= src.height
    ):
        raise RuntimeError(f'Jakarta ROI window falls outside GEOC raster dimensions: {source} ({source_window}).')
    source_roi_transform = src.window_transform(source_window)
    source_roi_bounds = list(window_bounds(source_window, src.transform))
    corner_errors_pixels = {}
    for label, col, row in (
        ('upper_left', 0, 0), ('upper_right', target_width, 0),
        ('lower_left', 0, target_height), ('lower_right', target_width, target_height),
    ):
        source_x, source_y = source_roi_transform * (col, row)
        target_x, target_y = roi_transform * (col, row)
        x_error_pixels = abs(source_x - target_x) / source_res_deg[0]
        y_error_pixels = abs(source_y - target_y) / source_res_deg[1]
        corner_errors_pixels[label] = {
            'x': x_error_pixels, 'y': y_error_pixels,
            'maximum_axis_error': max(x_error_pixels, y_error_pixels),
        }
    max_corner_error_pixels = max(item['maximum_axis_error'] for item in corner_errors_pixels.values())
    max_grid_normalization_pixels = max(max_corner_error_pixels, *window_rounding_errors)
    compatible = max_grid_normalization_pixels <= MAX_GRID_NORMALIZATION_PIXELS
    reference_transform = reference_grid[3]
    source_first_centre = src.transform * (0.5, 0.5)
    source_last_centre = src.transform * (src.width - 0.5, src.height - 0.5)
    reference_upper_left_outer = reference_transform * (0, 0)
    reference_lower_right_outer = reference_transform * (reference_grid[0], reference_grid[1])
    legacy_endpoint_errors_pixels = {
        'first_centre_x': abs(source_first_centre[0] - reference_upper_left_outer[0]) / source_res_deg[0],
        'first_centre_y': abs(source_first_centre[1] - reference_upper_left_outer[1]) / source_res_deg[1],
        'last_centre_x': abs(source_last_centre[0] - reference_lower_right_outer[0]) / source_res_deg[0],
        'last_centre_y': abs(source_last_centre[1] - reference_lower_right_outer[1]) / source_res_deg[1],
    }
    legacy_grid_repair_eligible = (
        pathlib.Path(source).name.endswith('.geo.unw.tif')
        and [src.width, src.height] == [reference_grid[0], reference_grid[1]]
        and math.isclose(src.transform.b, 0.0, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(src.transform.d, 0.0, rel_tol=0.0, abs_tol=1e-12)
        and max(legacy_endpoint_errors_pixels.values()) <= LEGACY_GRID_ENDPOINT_TOLERANCE_PIXELS
    )
    accepted = compatible or legacy_grid_repair_eligible
    diagnostics = {
        'shape': [src.height, src.width],
        'bounds': list(src.bounds),
        'transform': list(src.transform),
        'roi_window_col_row_width_height': [
            int(source_window.col_off), int(source_window.row_off),
            int(source_window.width), int(source_window.height),
        ],
        'source_roi_transform_before_normalization': list(source_roi_transform),
        'source_roi_bounds_before_normalization': source_roi_bounds,
        'window_rounding_errors_pixels': window_rounding_errors,
        'corner_errors_pixels': corner_errors_pixels,
        'max_grid_normalization_pixels': max_grid_normalization_pixels,
        'compatible_with_normalization_limit': compatible,
        'legacy_grid_endpoint_errors_pixels': legacy_endpoint_errors_pixels,
        'legacy_licsar_unw_grid_repair_eligible': legacy_grid_repair_eligible,
        'accepted_by_preflight': accepted,
        'grid_resolution_policy': ('bounded_metadata_normalization' if compatible else ('licsar_legacy_nearest_match' if legacy_grid_repair_eligible else 'rejected')),
    }
    if enforce_limit and not accepted:
        raise RuntimeError(f'GEOC raster grid differs from the fixed ROI target by {max_grid_normalization_pixels:.6g} pixels, above the {MAX_GRID_NORMALIZATION_PIXELS} pixel limit and outside the exact LiCSAR legacy UNW repair family: {source}; transform={list(src.transform)}; floating_offsets={[floating_col_off, floating_row_off]}; rounding_errors_pixels={window_rounding_errors}; source_roi_bounds={source_roi_bounds}; target_bounds={effective_roi_bounds}; corner_errors_pixels={corner_errors_pixels}.')
    return source_window, diagnostics

def crop_geotiff(source, destination):
    with rasterio.open(source) as src:
        source_window, diagnostics = source_window_for_target(src, source)
        direct_window_data = src.read(window=source_window)
        repair_applied = diagnostics['legacy_licsar_unw_grid_repair_eligible'] and not diagnostics['compatible_with_normalization_limit']
        if repair_applied:
            data = np.empty((src.count, target_height, target_width), dtype=direct_window_data.dtype)
            for band_index in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_index), destination=data[band_index - 1],
                    src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                    dst_transform=roi_transform, dst_crs=reference_grid[2], dst_nodata=src.nodata,
                    resampling=Resampling.nearest, init_dest_nodata=True,
                )
            if not np.array_equal(data, direct_window_data, equal_nan=True):
                changed = int(np.count_nonzero(~np.isclose(data, direct_window_data, rtol=0.0, atol=0.0, equal_nan=True)))
                raise RuntimeError(f'LiCSAR legacy nearest-grid repair changed {changed:,} ROI array values instead of reproducing the source window exactly: {source}.')
        else:
            data = direct_window_data
        diagnostics['legacy_licsar_grid_repair_applied'] = repair_applied
        diagnostics['array_values_changed_by_grid_repair'] = False
        diagnostics['licsar_grid_repair_commit'] = LICSAR_GRID_REPAIR_COMMIT if repair_applied else None
        if data.shape[1:] != (target_height, target_width):
            raise RuntimeError(f'Unexpected cropped raster shape for {source}: {data.shape}.')
        profile = src.profile.copy()
        profile.pop('blockxsize', None)
        profile.pop('blockysize', None)
        profile.update(
            width=target_width, height=target_height,
            transform=roi_transform, tiled=False,
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(destination, 'w', **profile) as dst:
            dst.write(data)
            dst.update_tags(**src.tags())
            if repair_applied:
                dst.update_tags(
                    LICSAR_LEGACY_GRID_REPAIR='nearest_match_to_reference_grid_array_values_verified_identical',
                    LICSAR_GRID_REPAIR_COMMIT=LICSAR_GRID_REPAIR_COMMIT,
                    LICSAR_GRID_REPAIR_URL=LICSAR_GRID_REPAIR_URL,
                    SOURCE_TRANSFORM_BEFORE_REPAIR=json.dumps(list(src.transform)),
                )
            for band in range(1, src.count + 1):
                dst.update_tags(band, **src.tags(band))
                if src.descriptions[band - 1]:
                    dst.set_band_description(band, src.descriptions[band - 1])
            dst.write_mask(src.dataset_mask(window=source_window))
        return diagnostics

def preflight_source_rasters():
    source_rasters = sorted(geoc_root.rglob('*.tif'))
    if not source_rasters:
        raise RuntimeError('The verified GEOC stack contains no GeoTIFFs to crop.')
    maxima = {
        'abs_delta_pixel_width_degrees': {'value': 0.0, 'source': None},
        'abs_delta_pixel_height_degrees': {'value': 0.0, 'source': None},
        'abs_rotation_b': {'value': 0.0, 'source': None},
        'abs_rotation_d': {'value': 0.0, 'source': None},
        'grid_normalization_pixels': {'value': 0.0, 'source': None},
    }
    def record_maximum(name, value, source):
        if value > maxima[name]['value']:
            maxima[name] = {'value': value, 'source': str(source)}
    for source in source_rasters:
        with rasterio.open(source) as src:
            record_maximum('abs_delta_pixel_width_degrees', abs(src.transform.a - reference_grid[3].a), source)
            record_maximum('abs_delta_pixel_height_degrees', abs(src.transform.e - reference_grid[3].e), source)
            record_maximum('abs_rotation_b', abs(src.transform.b), source)
            record_maximum('abs_rotation_d', abs(src.transform.d), source)
    transform_audit = {
        'raster_count': len(source_rasters),
        'reference_raster': str(source_unw),
        'reference_transform': list(reference_grid[3]),
        'transform_coefficient_maxima': {name: maxima[name] for name in (
            'abs_delta_pixel_width_degrees', 'abs_delta_pixel_height_degrees',
            'abs_rotation_b', 'abs_rotation_d',
        )},
    }
    print('ROI source-transform audit (before compatibility gating):')
    print(json.dumps(transform_audit, indent=2))
    raster_grid_diagnostics = []
    incompatible_rasters = []
    legacy_grid_repair_rasters = []
    hard_errors = []
    normalization_values = []
    for source in source_rasters:
        try:
            with rasterio.open(source) as src:
                _, diagnostics = source_window_for_target(src, source, enforce_limit=False)
        except Exception as exc:
            hard_errors.append({'source': str(source), 'pair': source.parent.name, 'error': str(exc)})
            continue
        value = diagnostics['max_grid_normalization_pixels']
        normalization_values.append(value)
        record_maximum('grid_normalization_pixels', value, source)
        record = {
            'source': str(source), 'pair': source.parent.name,
            'raster_type': 'unw' if source.name.endswith('.geo.unw.tif') else ('cc' if source.name.endswith('.geo.cc.tif') else 'other'),
            **diagnostics,
        }
        raster_grid_diagnostics.append(record)
        if diagnostics['legacy_licsar_unw_grid_repair_eligible'] and not diagnostics['compatible_with_normalization_limit']:
            legacy_grid_repair_rasters.append(record)
        elif not diagnostics['accepted_by_preflight']:
            incompatible_rasters.append(record)
    incompatible_rasters.sort(key=lambda item: item['max_grid_normalization_pixels'], reverse=True)
    legacy_grid_repair_rasters.sort(key=lambda item: item['source'])
    normalization_values.sort()
    def percentile(values, percent):
        if not values:
            return None
        position = (len(values) - 1) * percent / 100.0
        lower = math.floor(position)
        upper = math.ceil(position)
        if lower == upper:
            return values[lower]
        return values[lower] + (values[upper] - values[lower]) * (position - lower)
    incompatible_pairs = sorted({item['pair'] for item in incompatible_rasters + hard_errors})
    legacy_grid_repair_pairs = sorted({item['pair'] for item in legacy_grid_repair_rasters})
    incompatible_type_counts = {
        kind: sum(item.get('raster_type') == kind for item in incompatible_rasters)
        for kind in ('unw', 'cc', 'other')
    }
    summary = {
        'raster_count': len(source_rasters),
        'reference_raster': str(source_unw),
        'reference_transform': list(reference_grid[3]),
        'maximum_allowed_grid_normalization_pixels': MAX_GRID_NORMALIZATION_PIXELS,
        'maxima': maxima,
        'normalization_percentiles_pixels': {
            f'p{percent}': percentile(normalization_values, percent)
            for percent in (0, 25, 50, 75, 90, 95, 99, 100)
        },
        'compatible_raster_count': len(source_rasters) - len(legacy_grid_repair_rasters) - len(incompatible_rasters) - len(hard_errors),
        'legacy_grid_repair_raster_count': len(legacy_grid_repair_rasters),
        'legacy_grid_repair_pair_count': len(legacy_grid_repair_pairs),
        'expected_legacy_grid_repair_count': EXPECTED_LEGACY_UNW_REPAIRS,
        'accepted_raster_count': len(source_rasters) - len(incompatible_rasters) - len(hard_errors),
        'incompatible_raster_count': len(incompatible_rasters) + len(hard_errors),
        'incompatible_pair_count': len(incompatible_pairs),
        'incompatible_raster_type_counts': incompatible_type_counts,
        'hard_error_count': len(hard_errors),
        'top_incompatible_rasters': incompatible_rasters[:20],
        'hard_errors': hard_errors[:20],
        'full_diagnostic_path': str(GEOC_ROI_PREFLIGHT),
    }
    full_diagnostic = {
        **summary,
        'incompatible_pairs': incompatible_pairs,
        'incompatible_rasters': incompatible_rasters,
        'legacy_grid_repair_pairs': legacy_grid_repair_pairs,
        'legacy_grid_repair_rasters': legacy_grid_repair_rasters,
        'all_raster_grid_diagnostics': raster_grid_diagnostics,
        'all_hard_errors': hard_errors,
    }
    GEOC_ROI_PREFLIGHT.write_text(json.dumps(full_diagnostic, indent=2), encoding='utf-8')
    print('ROI source-grid preflight summary:')
    print(json.dumps(summary, indent=2))
    if incompatible_rasters or hard_errors:
        raise RuntimeError(f'ROI grid preflight rejected {len(incompatible_rasters) + len(hard_errors):,} of {len(source_rasters):,} source GeoTIFFs across {len(incompatible_pairs):,} pair directories; no raster was cropped or removed. Full diagnostics: {GEOC_ROI_PREFLIGHT}')
    if len(legacy_grid_repair_rasters) != EXPECTED_LEGACY_UNW_REPAIRS:
        raise RuntimeError(f'ROI grid preflight found {len(legacy_grid_repair_rasters):,} exact LiCSAR legacy UNW grids; expected {EXPECTED_LEGACY_UNW_REPAIRS:,}. No raster was cropped. Full diagnostics: {GEOC_ROI_PREFLIGHT}')
    print(f'ROI grid preflight accepted {len(source_rasters):,} source GeoTIFFs: {len(legacy_grid_repair_rasters):,} exact legacy LiCSAR UNWs require nearest-grid repair and every other raster remains inside the {MAX_GRID_NORMALIZATION_PIXELS} pixel metadata-normalization limit.')
    return summary

def build_roi_stack(preflight_summary):
    partial_root = WORK / f'{GEOC_ROI_NAME}.partial'
    if partial_root.exists():
        shutil.rmtree(partial_root)
    partial_root.mkdir(parents=True)
    cropped_rasters = 0
    copied_files = 0
    skipped_sidecars = 0
    source_grid_counts = {}
    maximum_grid_normalization_pixels = 0.0
    legacy_grid_repairs = 0
    for source in sorted(geoc_root.rglob('*')):
        relative = source.relative_to(geoc_root)
        destination = partial_root / relative
        if source.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        elif source.name.lower().endswith(('.tif.msk', '.tif.aux.xml', '.tif.ovr')):
            skipped_sidecars += 1
        elif source.suffix.lower() == '.tif':
            source_grid = crop_geotiff(source, destination)
            if source_grid['legacy_licsar_grid_repair_applied']:
                legacy_grid_repairs += 1
            else:
                maximum_grid_normalization_pixels = max(maximum_grid_normalization_pixels, source_grid['max_grid_normalization_pixels'])
            source_grid_key = json.dumps(source_grid, sort_keys=True)
            source_grid_counts[source_grid_key] = source_grid_counts.get(source_grid_key, 0) + 1
            cropped_rasters += 1
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            copied_files += 1
    source_grid_variants = []
    for source_grid_key, raster_count in sorted(source_grid_counts.items()):
        source_grid_variant = json.loads(source_grid_key)
        source_grid_variant['raster_count'] = raster_count
        source_grid_variants.append(source_grid_variant)
    provenance = {
        'roi_crop_schema': ROI_CROP_SCHEMA,
        'crop_policy': 'per_raster_integer_window_with_0.01px_metadata_limit_plus_exact_legacy_licsar_unw_nearest_match_verified_array_identical',
        'maximum_allowed_grid_normalization_pixels': MAX_GRID_NORMALIZATION_PIXELS,
        'maximum_applied_grid_normalization_pixels': maximum_grid_normalization_pixels,
        'legacy_licsar_grid_repairs': legacy_grid_repairs,
        'expected_legacy_licsar_grid_repairs': EXPECTED_LEGACY_UNW_REPAIRS,
        'legacy_grid_repair_method': 'nearest_neighbour_match_to_reference_grid_with_bitwise_equal_roi_array_assertion',
        'licsar_grid_repair_commit': LICSAR_GRID_REPAIR_COMMIT,
        'licsar_grid_repair_url': LICSAR_GRID_REPAIR_URL,
        'source_grid_preflight': preflight_summary,
        'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'frame': FRAME, 'start': START, 'end': END,
        'source_archive': str(GEOC_ARCHIVE),
        'source_archive_sha256': archive_sha,
        'source_reference_raster': str(source_unw),
        'source_bounds': source_bounds, 'source_shape': source_shape,
        'requested_roi_bounds_wsen': list(ROI_BOUNDS),
        'effective_pixel_aligned_bounds_wsen': effective_roi_bounds,
        'crop_window_col_row_width_height': [int(crop_window.col_off), int(crop_window.row_off), int(crop_window.width), int(crop_window.height)],
        'cropped_shape': [target_height, target_width],
        'crs': str(reference_grid[2]),
        'transform': list(roi_transform),
        'cropped_geotiffs': cropped_rasters, 'copied_non_raster_files': copied_files,
        'skipped_full_frame_raster_sidecars': skipped_sidecars,
        'source_grid_variant_count': len(source_grid_variants),
        'source_grid_variants': source_grid_variants,
    }
    (partial_root / 'roi_crop_provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
    os.replace(partial_root, geoc_roi_root)
    return provenance

ROI_RESTORED = False
if not geoc_roi_root.exists() and GEOC_ROI_ARCHIVE.exists() and GEOC_ROI_ARCHIVE_SHA.exists():
    expected_roi_sha = GEOC_ROI_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
    actual_roi_sha = sha256_file(GEOC_ROI_ARCHIVE)
    if actual_roi_sha != expected_roi_sha:
        raise RuntimeError('The Jakarta ROI checkpoint checksum does not match; refusing to restore it.')
    subprocess.run(['tar', '-xzf', str(GEOC_ROI_ARCHIVE), '-C', str(WORK)], check=True)
    ROI_RESTORED = True
    print('Restored verified Jakarta ROI checkpoint:', GEOC_ROI_ARCHIVE)

existing_roi = inventory_pairs(geoc_roi_root)
roi_provenance_path = geoc_roi_root / 'roi_crop_provenance.json'
if geoc_roi_root.exists() and (existing_roi['complete_pairs'] != EXPECTED_COMPLETE_PAIRS or not roi_provenance_path.exists()):
    print('Removing an incomplete or unprovenanced in-runtime ROI directory before rebuilding it.')
    shutil.rmtree(geoc_roi_root)
    ROI_RESTORED = False
if geoc_roi_root.exists():
    roi_provenance = json.loads(roi_provenance_path.read_text(encoding='utf-8'))
    provenance_mismatch = (
        roi_provenance.get('roi_crop_schema') != ROI_CROP_SCHEMA
        or roi_provenance.get('requested_roi_bounds_wsen') != list(ROI_BOUNDS)
        or roi_provenance.get('source_archive_sha256') != archive_sha
    )
    if provenance_mismatch:
        print('Removing an obsolete or mismatched ROI checkpoint before rebuilding it.')
        shutil.rmtree(geoc_roi_root)
        ROI_RESTORED = False
if not geoc_roi_root.exists():
    preflight_summary = preflight_source_rasters()
    roi_provenance = build_roi_stack(preflight_summary)
    if roi_provenance.get('cropped_geotiffs') != preflight_summary['raster_count']:
        raise RuntimeError('The ROI crop did not write every preflighted source GeoTIFF.')
    if roi_provenance.get('maximum_applied_grid_normalization_pixels') > MAX_GRID_NORMALIZATION_PIXELS:
        raise RuntimeError('The ROI crop exceeded the bounded grid-normalization limit.')
    if roi_provenance.get('legacy_licsar_grid_repairs') != EXPECTED_LEGACY_UNW_REPAIRS:
        raise RuntimeError('The ROI crop did not repair exactly the expected LiCSAR legacy UNW grid family.')

if roi_provenance.get('roi_crop_schema') != ROI_CROP_SCHEMA:
    raise RuntimeError('The ROI checkpoint uses an unsupported crop schema.')
if roi_provenance.get('requested_roi_bounds_wsen') != list(ROI_BOUNDS):
    raise RuntimeError('The ROI checkpoint does not use the fixed Jakarta bounds; refusing to continue.')
if roi_provenance.get('source_archive_sha256') != archive_sha:
    raise RuntimeError('The ROI checkpoint was not derived from the currently verified GEOC archive.')
if roi_provenance.get('legacy_licsar_grid_repairs') != EXPECTED_LEGACY_UNW_REPAIRS:
    raise RuntimeError('The ROI checkpoint does not contain the expected verified LiCSAR legacy-grid repairs.')
roi_inventory = require_complete_pairs(geoc_roi_root, GEOC_ROI_NAME, MIN_COMPLETE_PAIRS)
if roi_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'{GEOC_ROI_NAME} has {roi_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')
roi_inventory.update({'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'frame': FRAME, 'start': START, 'end': END, 'roi': roi_provenance})
GEOC_ROI_INVENTORY.write_text(json.dumps(roi_inventory, indent=2), encoding='utf-8')

if not ROI_RESTORED:
    partial_roi = GEOC_ROI_ARCHIVE.with_suffix(GEOC_ROI_ARCHIVE.suffix + '.partial')
    roi_archive_run = subprocess.run(['tar', '-czf', str(partial_roi), '-C', str(WORK), GEOC_ROI_NAME])
    if roi_archive_run.returncode != 0 or not partial_roi.exists() or partial_roi.stat().st_size == 0:
        raise RuntimeError('Jakarta ROI checkpoint creation failed; do not continue.')
    os.replace(partial_roi, GEOC_ROI_ARCHIVE)
roi_checkpoint_sha = sha256_file(GEOC_ROI_ARCHIVE)
GEOC_ROI_ARCHIVE_SHA.write_text(f'{roi_checkpoint_sha}  {GEOC_ROI_ARCHIVE.name}\n', encoding='utf-8')
roi_listing = subprocess.run(['tar', '-tzf', str(GEOC_ROI_ARCHIVE)], check=True, text=True, stdout=subprocess.PIPE).stdout.splitlines()
roi_archive_unw = {pathlib.PurePosixPath(item).parts[1] for item in roi_listing if item.endswith('.geo.unw.tif') and len(pathlib.PurePosixPath(item).parts) >= 3}
roi_archive_cc = {pathlib.PurePosixPath(item).parts[1] for item in roi_listing if item.endswith('.geo.cc.tif') and len(pathlib.PurePosixPath(item).parts) >= 3}
if roi_archive_unw != roi_archive_cc or len(roi_archive_unw) != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'ROI archive pair mismatch: UNW={len(roi_archive_unw)}, CC={len(roi_archive_cc)}.')
print(f'VERIFIED ROI checkpoint: {GEOC_ROI_ARCHIVE} ({len(roi_archive_unw):,} complete pairs)')
print('ROI checkpoint SHA-256:', roi_checkpoint_sha)

GEOCML_RESTORED = False
geocml_root = WORK / GEOCML_NAME
if not geocml_root.exists() and GEOCML_ARCHIVE.exists() and GEOCML_ARCHIVE_SHA.exists():
    expected_ml_sha = GEOCML_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
    actual_ml_sha = sha256_file(GEOCML_ARCHIVE)
    if actual_ml_sha != expected_ml_sha:
        raise RuntimeError('The multilook checkpoint checksum does not match; refusing to restore it.')
    subprocess.run(['tar', '-xzf', str(GEOCML_ARCHIVE), '-C', str(WORK)], check=True)
    GEOCML_RESTORED = True
    print('Restored verified multilook checkpoint:', GEOCML_ARCHIVE)

existing_ml = inventory_pairs(geocml_root)
if existing_ml['pair_folders'] and existing_ml['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    print('Removing an incomplete in-runtime multilook directory before rebuilding it.')
    shutil.rmtree(geocml_root)
    GEOCML_RESTORED = False

if not geocml_root.exists():
    run_script(
        'multilook', ['-i', GEOC_ROI_NAME, '-o', GEOCML_NAME, '-n', NLOOK],
        required=[geocml_root],
    )
else:
    print(f'{GEOCML_NAME} already exists; validating it instead of repeating step 02.')

geocml_inventory = require_complete_pairs(geocml_root, GEOCML_NAME, MIN_COMPLETE_PAIRS)
if geocml_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'{GEOCML_NAME} has {geocml_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')

dem_values = {}
for line in (geocml_root / 'EQA.dem_par').read_text(encoding='utf-8', errors='replace').splitlines():
    fields = line.replace(':', ' ').split()
    if len(fields) >= 2 and fields[0] in {'width', 'nlines', 'post_lat', 'post_lon'}:
        dem_values[fields[0]] = fields[1]
resolution_summary = {
    'source_file': str(source_unw),
    'full_frame_source_shape': source_shape,
    'roi_source_shape': [target_height, target_width],
    'source_resolution_degrees': source_res_deg,
    'source_resolution_metres_approx': [source_x_m, source_y_m],
    'multilook_factor': NLOOK,
    'effective_resolution_metres_approx': [source_x_m * NLOOK, source_y_m * NLOOK],
    'requested_roi_bounds_wsen': list(ROI_BOUNDS),
    'effective_pixel_aligned_bounds_wsen': effective_roi_bounds,
    'multilooked_parameters': dem_values,
}
geocml_inventory.update({
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'start': START, 'end': END, 'run_tag': RUN_TAG,
    'roi': roi_provenance, 'resolution': resolution_summary,
})
GEOCML_INVENTORY.write_text(json.dumps(geocml_inventory, indent=2), encoding='utf-8')
print(json.dumps(resolution_summary, indent=2))

if not GEOCML_RESTORED:
    partial_ml = GEOCML_ARCHIVE.with_suffix(GEOCML_ARCHIVE.suffix + '.partial')
    checkpoint_run = subprocess.run(['tar', '-czf', str(partial_ml), '-C', str(WORK), GEOCML_NAME])
    if checkpoint_run.returncode != 0 or not partial_ml.exists() or partial_ml.stat().st_size == 0:
        raise RuntimeError('Multilook checkpoint creation failed; do not continue.')
    os.replace(partial_ml, GEOCML_ARCHIVE)
checkpoint_sha = sha256_file(GEOCML_ARCHIVE)
GEOCML_ARCHIVE_SHA.write_text(f'{checkpoint_sha}  {GEOCML_ARCHIVE.name}\n', encoding='utf-8')
subprocess.run(['tar', '-tzf', str(GEOCML_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
print(f'VERIFIED {RUN_TAG} checkpoint: {GEOCML_ARCHIVE} ({GEOCML_ARCHIVE.stat().st_size / 1e9:.2f} GB)')
print('Checkpoint SHA-256:', checkpoint_sha)


In [ ]:
# 9 — Steps 11 and 12: quality check, multi-prime loop closure, and network gate
run_script('check', ['-d', GEOCML_NAME, '-t', TS_NAME], required=[WORK / TS_NAME])
run_script(
    'loop', ['-d', GEOCML_NAME, '-t', TS_NAME, '-l', str(LOOP_THRESHOLD_RAD), '--multi_prime'],
    required=[WORK / TS_NAME / 'info' / '12bad_ifg.txt'],
)

def data_lines(path):
    path = pathlib.Path(path)
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip() and not line.lstrip().startswith('#')]

def interferogram_ids(lines, label):
    parsed = []
    unparsed = []
    for line in lines:
        match = re.search(r'(?<!\d)(\d{8}_\d{8})(?!\d)', line)
        if match:
            parsed.append(match.group(1))
        else:
            unparsed.append(line)
    if unparsed:
        raise RuntimeError(f'Cannot parse interferogram IDs from {label}: {unparsed[:5]}')
    return set(parsed)

info_dir = WORK / TS_NAME / 'info'
bad11_lines = data_lines(info_dir / '11bad_ifg.txt')
bad12_lines = data_lines(info_dir / '12bad_ifg.txt')
removed12 = data_lines(info_dir / '12removed_image.txt')
bad11 = interferogram_ids(bad11_lines, '11bad_ifg.txt')
bad12 = interferogram_ids(bad12_lines, '12bad_ifg.txt')
all_ifgs = sorted(path.name for path in (WORK / GEOCML_NAME).iterdir() if path.is_dir() and PAIR_RE.fullmatch(path.name))
if len(all_ifgs) != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Network gate found {len(all_ifgs)} interferograms; expected {EXPECTED_COMPLETE_PAIRS}.')
all_ifg_set = set(all_ifgs)
unknown_bad = (bad11 | bad12) - all_ifg_set
if unknown_bad:
    raise RuntimeError(f'Bad-interferogram files reference unknown pairs: {sorted(unknown_bad)[:5]}')
rejected_ifgs = bad11 | bad12
retained_ifgs = sorted(all_ifg_set - rejected_ifgs)
bad_fraction = len(rejected_ifgs) / len(all_ifgs)
all_images = sorted({date for ifg in all_ifgs for date in ifg.split('_')})
adjacency = {date: set() for date in all_images}
for ifg in retained_ifgs:
    first, second = ifg.split('_')
    adjacency[first].add(second)
    adjacency[second].add(first)

components = []
unvisited = set(all_images)
while unvisited:
    start_image = min(unvisited)
    stack = [start_image]
    component = set()
    while stack:
        image = stack.pop()
        if image in component:
            continue
        component.add(image)
        unvisited.discard(image)
        stack.extend(sorted(adjacency[image] - component, reverse=True))
    components.append(sorted(component))
components.sort(key=lambda component: (-len(component), component[0]))
largest_component = components[0] if components else []
requested_days = (datetime.datetime.strptime(END, '%Y%m%d') - datetime.datetime.strptime(START, '%Y%m%d')).days
if len(largest_component) >= 2:
    component_days = (datetime.datetime.strptime(largest_component[-1], '%Y%m%d') - datetime.datetime.strptime(largest_component[0], '%Y%m%d')).days
    component_span_fraction = component_days / requested_days
else:
    component_span_fraction = 0.0
component_image_fraction = len(largest_component) / len(all_images) if all_images else 0.0
component_summaries = []
for component in components:
    component_ifgs = [ifg for ifg in retained_ifgs if set(ifg.split('_')).issubset(component)]
    span_days = (datetime.datetime.strptime(component[-1], '%Y%m%d') - datetime.datetime.strptime(component[0], '%Y%m%d')).days if len(component) >= 2 else 0
    component_summaries.append({
        'image_count': len(component), 'interferogram_count': len(component_ifgs),
        'first_image': component[0], 'last_image': component[-1],
        'temporal_span_days': span_days,
        'temporal_span_fraction_of_requested_period': span_days / requested_days,
        'images': component,
    })
network_gate = {
    'gate_basis': 'largest_connected_component_of_retained_interferograms',
    'loop_mode': LOOP_MODE,
    'loop_threshold_rad': LOOP_THRESHOLD_RAD,
    'total_interferograms': len(all_ifgs),
    'step11_bad_interferograms': len(bad11),
    'step12_bad_interferograms': len(bad12),
    'unique_rejected_interferograms': len(rejected_ifgs),
    'retained_interferograms': len(retained_ifgs),
    'bad_interferogram_fraction': bad_fraction,
    'total_images': len(all_images),
    'step12_reported_removed_images': removed12,
    'connected_component_count_including_isolates': len(components),
    'largest_connected_component': {
        'images': largest_component,
        'image_count': len(largest_component),
        'image_fraction_of_all_acquisitions': component_image_fraction,
        'first_image': largest_component[0] if largest_component else None,
        'last_image': largest_component[-1] if largest_component else None,
        'temporal_span_fraction_of_requested_period': component_span_fraction,
    },
    'components': component_summaries,
    'thresholds': {
        'max_loop_bad_fraction': MAX_LOOP_BAD_FRACTION,
        'min_largest_component_image_fraction': MIN_RETAINED_IMAGE_FRACTION,
        'min_largest_component_temporal_span_fraction': MIN_TEMPORAL_SPAN_FRACTION,
    },
    'pass_action': 'Run factor 2 over the identical fixed ROI and identical gates.',
    'fail_action': 'Stop LiCSBAS parameter experiments and choose alternative InSAR processing/data or a regional diagnostic scope.',
}
NETWORK_GATE_PATH = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_network_gate.json'
NETWORK_GATE_PATH.write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
NETWORK_GATE_WORK_PATH = info_dir / f'{RUN_TAG}_network_gate.json'
NETWORK_GATE_WORK_PATH.write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
(info_dir / f'{RUN_TAG}_roi_crop_provenance.json').write_text(json.dumps(roi_provenance, indent=2), encoding='utf-8')
print(json.dumps(network_gate, indent=2))
gate_failures = []
if bad_fraction > MAX_LOOP_BAD_FRACTION:
    gate_failures.append(f'loop closure rejected {bad_fraction:.1%} of interferograms')
if component_image_fraction < MIN_RETAINED_IMAGE_FRACTION:
    gate_failures.append(f'largest connected component contains only {component_image_fraction:.1%} of acquisition images')
if component_span_fraction < MIN_TEMPORAL_SPAN_FRACTION:
    gate_failures.append(f'largest connected component spans only {component_span_fraction:.1%} of the requested period')
DIAGNOSTIC_ARCHIVE = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.tar.gz'
DIAGNOSTIC_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.sha256'
diagnostic_items = [
    f'{TS_NAME}/info',
    f'{TS_NAME}/network',
    f'{TS_NAME}/12loop/loop_info.txt',
    f'{TS_NAME}/12loop/loop_ph_rms_masked.png',
    f'{TS_NAME}/12loop/bad_ifg_loop.txt',
    f'{TS_NAME}/12loop/bad_ifg_loopref.txt',
    f'{TS_NAME}/12loop/bad_ifg_noref.txt',
]
existing_diagnostic_items = [item for item in diagnostic_items if (WORK / item).exists()]
partial_diagnostic = DIAGNOSTIC_ARCHIVE.with_suffix(DIAGNOSTIC_ARCHIVE.suffix + '.partial')
diagnostic_run = subprocess.run(
    ['tar', '-czf', str(partial_diagnostic), '-C', str(WORK), *existing_diagnostic_items]
)
if diagnostic_run.returncode != 0 or not partial_diagnostic.exists() or partial_diagnostic.stat().st_size == 0:
    raise RuntimeError('Step-12 diagnostic archive failed; do not continue.')
os.replace(partial_diagnostic, DIAGNOSTIC_ARCHIVE)
subprocess.run(['tar', '-tzf', str(DIAGNOSTIC_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
diagnostic_sha = sha256_file(DIAGNOSTIC_ARCHIVE)
DIAGNOSTIC_ARCHIVE_SHA.write_text(f'{diagnostic_sha}  {DIAGNOSTIC_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED step-12 diagnostics: {DIAGNOSTIC_ARCHIVE} ({DIAGNOSTIC_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('Diagnostic SHA-256:', diagnostic_sha)

if gate_failures:
    raise RuntimeError('NETWORK SURVIVAL GATE FAILED: ' + '; '.join(gate_failures) + '. Diagnostics are safely archived in Drive; do not continue to inversion.')
print('NETWORK SURVIVAL GATE PASSED — safe to continue to step 13.')


In [ ]:
# 10 — Steps 13 and 14: SBAS inversion and velocity uncertainty
run_script(
    'invert', ['-d', GEOCML_NAME, '-t', TS_NAME, '--mem_size', '4000'],
    required=[WORK / TS_NAME / 'cum.h5', WORK / TS_NAME / 'results' / 'vel'],
)
run_script(
    'vstd', ['-t', TS_NAME],
    required=[WORK / TS_NAME / 'results' / 'vstd'],
)


In [ ]:
# 11 — Export the matched four-layer GeoTIFF stack with exact dates and factor label
results_dir = WORK / TS_NAME / 'results'
dem_par = WORK / GEOCML_NAME / 'EQA.dem_par'
layers = ('vel', 'vstd', 'coh_avg', 'n_unw')
exported = {}
for layer in layers:
    source = results_dir / layer
    if not source.exists():
        raise FileNotFoundError(f'Missing LiCSBAS result layer: {source}')
    destination = DRIVE_OUT / f'jakarta_{layer}_{DATE_TAG}_{RUN_TAG}.tif'
    run_script(
        'geotiff', ['-i', source, '-p', dem_par, '-o', destination],
        required=[destination],
    )
    exported[layer] = destination
print(json.dumps({k: str(v) for k, v in exported.items()}, indent=2))


In [ ]:
# 12 — Validate grids/values and write provenance plus a scientific QC summary
import numpy as np
import rasterio

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

layer_meta = {}
grid_key = None
for role, path in exported.items():
    with rasterio.open(path) as src:
        data = src.read(1)
        current_grid = (src.width, src.height, str(src.crs), tuple(src.transform))
        if grid_key is None:
            grid_key = current_grid
        elif current_grid != grid_key:
            raise RuntimeError(f'Grid mismatch for {role}: {current_grid} != {grid_key}')
        finite = np.isfinite(data)
        values = data[finite]
        if role == 'vstd' and (not finite.any() or np.any(values < 0)):
            raise RuntimeError('vstd is empty or contains negative values')
        nodata_value = src.nodata
        if isinstance(nodata_value, float) and not np.isfinite(nodata_value):
            nodata_value = None
        layer_meta[role] = {
            'path': str(path), 'sha256': sha256(path), 'bytes': path.stat().st_size,
            'shape': [src.height, src.width], 'crs': str(src.crs),
            'transform': list(src.transform), 'resolution': list(src.res),
            'nodata': nodata_value,
            'total_pixels': int(data.size),
            'finite_pixels': int(finite.sum()),
            'finite_fraction': float(finite.mean()),
            'min': float(values.min()) if values.size else None,
            'median': float(np.median(values)) if values.size else None,
            'max': float(values.max()) if values.size else None,
        }

with rasterio.open(exported['vel']) as vel_src, rasterio.open(exported['vstd']) as vstd_src:
    common_valid = np.isfinite(vel_src.read(1)) & np.isfinite(vstd_src.read(1))

def count_data_lines(path):
    if path is None or not path.exists():
        return None
    return sum(1 for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip() and not line.lstrip().startswith('#'))

info_dir = WORK / TS_NAME / 'info'
def first_info(pattern):
    matches = sorted(info_dir.glob(pattern)) if info_dir.exists() else []
    return matches[0] if matches else None

licsbas_network = {
    'step11_bad_interferograms': count_data_lines(first_info('11bad_ifg*.txt')),
    'step12_bad_interferograms': count_data_lines(first_info('12bad_ifg*.txt')),
    'step13_used_images': count_data_lines(first_info('13used_image*.txt')),
}
quality_flags = []
vel_meta = layer_meta['vel']
vstd_meta = layer_meta['vstd']
coh_meta = layer_meta['coh_avg']
n_unw_meta = layer_meta['n_unw']
if vel_meta['finite_fraction'] < 0.50:
    quality_flags.append('velocity coverage is below 50% of the output grid')
if coh_meta['median'] is not None and coh_meta['median'] < 0.20:
    quality_flags.append('median average coherence is below 0.20')
if vstd_meta['median'] is not None and vstd_meta['median'] >= 12.0:
    quality_flags.append('median vstd is close to or above the planned 15 mm/year threshold')
if vel_meta['min'] is not None and max(abs(vel_meta['min']), abs(vel_meta['max'])) > 200.0:
    quality_flags.append('absolute velocity extremes exceed 200 mm/year and require validation')
if n_unw_meta['max'] is not None and n_unw_meta['max'] < 20:
    quality_flags.append('maximum contributing unwrapped-interferogram count is below 20')

qc = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'scientific_status': 'diagnostic_candidate_pending_validation',
    'run_role': RUN_ROLE,
    'frame': FRAME, 'orbit': 'ascending', 'start': START, 'end': END,
    'multilook': NLOOK, 'run_tag': RUN_TAG, 'loop_mode': LOOP_MODE,
    'roi': roi_provenance,
    'common_valid_vel_vstd_pixels': int(common_valid.sum()),
    'common_valid_vel_vstd_fraction': float(common_valid.mean()),
    'resolution': resolution_summary,
    'network': licsbas_network,
    'network_survival_gate': network_gate,
    'layer_statistics': layer_meta,
    'quality_flags': quality_flags,
    'interpretation': 'A passed network gate would authorize only the identical fixed-ROI factor-2 test; it would not validate a hazard layer. A failed gate ends LiCSBAS parameter experiments. Any candidate still requires uncertainty, coherence, reference, and external validation before promotion.',
}
QC_SUMMARY.write_text(json.dumps(qc, indent=2, allow_nan=False), encoding='utf-8')

manifest = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'orbit': 'ascending', 'start': START, 'end': END,
    'multilook': NLOOK, 'run_tag': RUN_TAG, 'run_role': RUN_ROLE, 'loop_mode': LOOP_MODE,
    'licsbas_commit': actual_commit,
    'input_archive': {
        'path': str(GEOC_ARCHIVE), 'sha256': archive_sha,
        'complete_pairs': geoc_inventory['complete_pairs'],
    },
    'roi_checkpoint': {
        'path': str(GEOC_ROI_ARCHIVE), 'sha256': roi_checkpoint_sha,
        'complete_pairs': roi_inventory['complete_pairs'], 'roi': roi_provenance,
    },
    'multilook_checkpoint': {
        'path': str(GEOCML_ARCHIVE), 'sha256': checkpoint_sha,
        'complete_pairs': geocml_inventory['complete_pairs'],
    },
    'resolution': resolution_summary,
    'network': licsbas_network,
    'network_survival_gate': network_gate,
    'steps': ['01', 'fixed_roi_crop_before_multilook', '02', '11', '12', '13', '14'],
    'layers': layer_meta,
    'qc_summary': str(QC_SUMMARY),
}
manifest_path = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps(qc, indent=2))
print('QC summary:', QC_SUMMARY)
print('Manifest:', manifest_path)
if quality_flags:
    print('QC flags were recorded. The run is a diagnostic candidate, not a validated hazard layer.')


In [ ]:
# 13 — Archive final results and verify them
RESULTS_ARCHIVE = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.tar.gz'
RESULTS_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.sha256'
items = [f'{TS_NAME}/results', f'{TS_NAME}/info', f'{TS_NAME}/network', f'{GEOC_ROI_NAME}/roi_crop_provenance.json', f'{GEOCML_NAME}/EQA.dem_par']
existing_items = [item for item in items if (WORK / item).exists()]
archive_run = subprocess.run(
    ['tar', '-czf', str(RESULTS_ARCHIVE), '-C', str(WORK), *existing_items]
)
if archive_run.returncode != 0 or not RESULTS_ARCHIVE.exists() or RESULTS_ARCHIVE.stat().st_size == 0:
    raise RuntimeError('Final-results archive failed')
subprocess.run(['tar', '-tzf', str(RESULTS_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
results_sha = sha256_file(RESULTS_ARCHIVE)
RESULTS_ARCHIVE_SHA.write_text(f'{results_sha}  {RESULTS_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED final archive: {RESULTS_ARCHIVE} ({RESULTS_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('Final archive SHA-256:', results_sha)
print(f'DONE: exact-date {RUN_TAG} GeoTIFFs, manifest, QC summary, and verified archives are in Drive.')
